In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from rasterio.features import geometry_mask

from data_process.n0_1o1_get_ge_sentinel2 import DIR_METADATA

In [ ]:
# ==========================================================
# Directories
# ==========================================================
DIR_DATA = Path("data")
DIR_METADATA = DIR_DATA / "0_metadata"
DIR_NDVI = DIR_DATA / "2_processed" / "2_Sentinel2"
DIR_MASK = DIR_DATA / "1_org" / "a_mask"

FILEPATH_NDVI_LIST = DIR_METADATA / "sentinel2-list.csv"
FILEPATH_PATCH_CENTERS = DIR_METADATA / "patch-centers.csv"

In [ ]:
def create_controlled_patch(
    target_path,
    background_path,
    mask_gdf,
    output_path,
    mask_output_path=None,
):
    """
    Create a controlled SAR patch by combining a target image with a
    background image using a polygon mask.

    Pixels inside the polygon are taken from the target image.
    Pixels outside the polygon are taken from the background image.
    """

    # ---------------------------------------
    # Read background SAR
    # ---------------------------------------
    with rasterio.open(background_path) as src:
        background = src.read(1)

    # ---------------------------------------
    # Read target SAR
    # ---------------------------------------
    with rasterio.open(target_path) as src:
        target = src.read(1)

        profile = src.profile.copy()
        transform = src.transform
        crs = src.crs
        height = src.height
        width = src.width

    # ---------------------------------------
    # Make sure polygon uses SAR CRS
    # ---------------------------------------
    mask_gdf = mask_gdf.to_crs(crs)

    # ---------------------------------------
    # Rasterize polygon on exact SAR grid
    # ---------------------------------------
    mask = geometry_mask(
        mask_gdf.geometry,
        out_shape=(height, width),
        transform=transform,
        invert=True,
    )

    # mask:
    # True  = inside polygon
    # False = outside polygon

    # ---------------------------------------
    # Create controlled image
    # ---------------------------------------
    controlled = np.where(
        mask,
        target,
        background,
    )

    # ---------------------------------------
    # Save controlled SAR
    # ---------------------------------------
    with rasterio.open(
        output_path,
        "w",
        **profile,
    ) as dst:
        dst.write(controlled, 1)

    # ---------------------------------------
    # Optionally save binary mask
    # ---------------------------------------
    if mask_output_path is not None:

        mask_profile = profile.copy()

        mask_profile.update(
            dtype="uint8",
            count=1,
            nodata=0,
        )

        with rasterio.open(
            mask_output_path,
            "w",
            **mask_profile,
        ) as dst:
            dst.write(
                mask.astype(np.uint8),
                1,
            )

    return controlled, mask

In [ ]:
# ==========================================================
# Read metadata
# ==========================================================
ndvi_df = pd.read_csv(FILEPATH_NDVI_LIST)
patch_df = pd.read_csv(FILEPATH_PATCH_CENTERS)

ndvi_df = ndvi_df[
    ndvi_df["control"] == True
].copy()

patch_df = patch_df[
    patch_df["control"] == True
].copy()

print(f"{len(ndvi_df)} controlled NDVI images")
print(f"{len(patch_df)} controlled patches")

# ==========================================================
# Cache polygons
# ==========================================================
mask_cache = {}

# ==========================================================
# Loop over controlled Sentinel-2 images
# ==========================================================
for _, ndvi_row in ndvi_df.iterrows():
    image_id = ndvi_row["name"]
    background_directory = ndvi_row["control_background"]
    print(f"\nProcessing image {image_id}")

    # ------------------------------------------------------
    # Construct image directories
    # ------------------------------------------------------
    year = pd.to_datetime(ndvi_row["date"]).year
    patch_root = DIR_NDVI / f"patches_{year}"

    image_dir = (
        patch_root
        / image_id
    )

    background_dir = (
        patch_root
        / background_directory
    )

    # ------------------------------------------------------
    # Loop over controlled patches
    # ------------------------------------------------------
    for _, patch_row in patch_df.iterrows():

        patch_id = patch_row["id"]
        area = patch_row["elora_area_id"]

        # --------------------------------------------------
        # Read polygon only once
        # --------------------------------------------------

        if area not in mask_cache:

            shp = (
                DIR_MASK
                / area
                / f"{area}.shp"
            )

            mask_cache[area] = gpd.read_file(shp)

        mask_gdf = mask_cache[area]

        # --------------------------------------------------
        # Construct file paths
        # --------------------------------------------------
        target_path = (
            image_dir
            / f"{image_id}_{patch_id}_NDVI.tif"
        )
        if not target_path.exists():
            continue

        background_path = (
            background_dir
            / f"{background_directory}_{patch_id}_NDVI.tif"
        )
        if not background_path.exists():
            print(f"Background patch not found: {background_path.name}")
            continue

        output_path = (
            image_dir
            / f"{image_id}_{patch_id}_NDVI_controlled.tif"
        )
        # --------------------------------------------------
        # Create controlled patch
        # --------------------------------------------------
        create_controlled_patch(
            target_path=target_path,
            background_path=background_path,
            mask_gdf=mask_gdf,
            output_path=output_path,
        )

print("Done.")